In [1]:
import os
import sys

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [2]:
print(sys.executable)

c:\Users\Dell\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder
    .appName("Spark")
    .master("local[*]")
    .getOrCreate()
)

In [5]:
spark

In [6]:
emp_data = [
    ("E101", "D01", "Amit Sharma", "Male", "75000", "2021-03-15"),
    ("E102", "D02", "Priya Mehta", "Female", "82000", "2020-07-21"),
    ("E103", "D01", "Rahul Verma", "Male", "68000", "2022-01-10"),
    ("E104", "D03", "Neha Kapoor", "Female", "91000", "2019-11-05"),
    ("E105", "D02", "Karan Singh", "Male", "72000", "2023-02-14"),
    ("E106", "D04", "Ananya Gupta", "Female", "88000", "2021-08-30"),
    ("E107", "D03", "Vikram Patel", "Male", "95000", "2018-12-12"),
    ("E108", "D01", "Sneha Reddy", "Female", "77000", "2022-06-19"),
    ("E109", "D04", "Arjun Nair", "Male", "83000", "2020-04-27"),
    ("E110", "D02", "Meera Iyer", "Female", "79000", "2023-09-01"),
]

emp_schema = "employee_id string, department_id string, name string, gender string, salary string, hire_date string"

In [7]:
emp =  spark.createDataFrame(data=emp_data, schema=emp_schema)

In [8]:
emp.rdd.getNumPartitions()

8

In [9]:
emp.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [10]:
emp_final = emp.where("salary > 50000")

In [11]:
emp_final.rdd.getNumPartitions()

8

In [15]:
emp_final.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("data/emp_output")

In [16]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [17]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [19]:
schema_string = "name string, age int"
schema_spark = StructType([
    StructField("name",StringType(), True),
    StructField("age",IntegerType(), True)
]) 

In [21]:
from pyspark.sql.functions import col, expr

emp['salary']

Column<'salary'>

In [26]:
emp_filter = emp.select(col('employee_id'), expr("name"), emp.salary)

In [27]:
emp_filter.show()

+-----------+------------+------+
|employee_id|        name|salary|
+-----------+------------+------+
|       E101| Amit Sharma| 75000|
|       E102| Priya Mehta| 82000|
|       E103| Rahul Verma| 68000|
|       E104| Neha Kapoor| 91000|
|       E105| Karan Singh| 72000|
|       E106|Ananya Gupta| 88000|
|       E107|Vikram Patel| 95000|
|       E108| Sneha Reddy| 77000|
|       E109|  Arjun Nair| 83000|
|       E110|  Meera Iyer| 79000|
+-----------+------------+------+



In [30]:
emp_filter_rename = emp.select(col('employee_id').alias("emp_id"),emp.name, expr("cast(salary as int)as salary"))

In [31]:
emp_filter_rename.explain(True)

== Parsed Logical Plan ==
'Project ['employee_id AS emp_id#117, name#2, cast('salary as int) AS salary#118]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Analyzed Logical Plan ==
emp_id: string, name: string, salary: int
Project [employee_id#0 AS emp_id#117, name#2, cast(salary#4 as int) AS salary#118]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Optimized Logical Plan ==
Project [employee_id#0 AS emp_id#117, name#2, cast(salary#4 as int) AS salary#118]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Physical Plan ==
*(1) Project [employee_id#0 AS emp_id#117, name#2, cast(salary#4 as int) AS salary#118]
+- *(1) Scan ExistingRDD[employee_id#0,department_id#1,name#2,gender#3,salary#4,hire_date#5]



In [32]:
emp_filter_rename.show()

+------+------------+------+
|emp_id|        name|salary|
+------+------------+------+
|  E101| Amit Sharma| 75000|
|  E102| Priya Mehta| 82000|
|  E103| Rahul Verma| 68000|
|  E104| Neha Kapoor| 91000|
|  E105| Karan Singh| 72000|
|  E106|Ananya Gupta| 88000|
|  E107|Vikram Patel| 95000|
|  E108| Sneha Reddy| 77000|
|  E109|  Arjun Nair| 83000|
|  E110|  Meera Iyer| 79000|
+------+------------+------+



In [33]:
emp_filter_rename.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [35]:
emp_filter_1 = emp_filter.selectExpr("employee_id as emp_id"," name", "cast(salary as int) as salary")
emp_filter_1.show()

+------+------------+------+
|emp_id|        name|salary|
+------+------------+------+
|  E101| Amit Sharma| 75000|
|  E102| Priya Mehta| 82000|
|  E103| Rahul Verma| 68000|
|  E104| Neha Kapoor| 91000|
|  E105| Karan Singh| 72000|
|  E106|Ananya Gupta| 88000|
|  E107|Vikram Patel| 95000|
|  E108| Sneha Reddy| 77000|
|  E109|  Arjun Nair| 83000|
|  E110|  Meera Iyer| 79000|
+------+------------+------+



In [36]:
emp_filter_1.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [45]:
emp_salary_filter = emp.selectExpr("*").where("salary>75000")

In [46]:
emp_salary_filter.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [51]:
emp_salary_filter.write.format("csv").mode("overwrite").save("data/output/emp_salary_filter.csv")

In [52]:
emp.write.format("csv").mode("overwrite").save("data/output/emp.csv")